# Step 5 — Lexicon-based pre-annotation of all 1,500 advertisements (schema v0.2)

**Purpose.** Apply the v0.2 schema to the full preprocessed dataset as a pre-annotation, and package it with the full advertisement for context so that the candidates can be reviewed later, in a separate study. The annotation guideline (`annotation-guidelines-gender-v0.2.md`) is the reference for every label.

**Decisions followed (from the project owner)**
- No human review at this stage; the candidates are kept as produced by the rules.
- Implicit candidates (I1, I2) and other uncertain candidates are marked `for_review` and the reason is written in `notes`; they are not removed.
- The full advertisement accompanies every candidate (`ad_context.csv`).

**Inputs.** `outputs/analysis/units.csv` (Steps 1–3) and `outputs/s4_schema_v0.2/crosswalk_step8e_to_v0.2.csv` (Step 4: Step 8e lexicon/rule spans translated to v0.2).

**Outputs** (`outputs/s5_preannotation/`)
- `preannotation_spans.csv` — one row per candidate span: label, family, review status, notes, unit offsets and raw offsets.
- `preannotation_units.csv` — every unit with its status and notes.
- `ad_context.csv` — every advertisement as masked text with component tags and its candidate summary.
- `validation_checks.csv`, `status_summary.csv`.

**What this is not.** The labels are rule-derived (silver) candidates. They are not human annotations, no agreement is measured, and counts here describe what the rules flag, not how much bias exists in the dataset. Employer names, contact details and URLs are masked as in Steps 1–3; no raw text is read or written.

In [1]:
from pathlib import Path
import json
import re

import pandas as pd

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 220)

OUT = Path("outputs") / "s5_preannotation"
OUT.mkdir(parents=True, exist_ok=True)

units = pd.read_csv("outputs/analysis/units.csv", encoding="utf-8-sig")
ads = pd.read_csv("outputs/analysis/ads.csv", encoding="utf-8-sig")
cw = pd.read_csv("outputs/s4_schema_v0.2/crosswalk_step8e_to_v0.2.csv", encoding="utf-8-sig")
schema = {s["code"]: s for s in json.load(open("outputs/s4_schema_v0.2/schema_v0.2.json", encoding="utf-8"))["labels"]}
print(f"{ads['record_id'].nunique():,} ads, {len(units):,} units, {len(cw):,} crosswalk span rows.")

1,500 ads, 7,971 units, 1,218 crosswalk span rows.


## 1. Review status and notes for each candidate span

| Condition | `review_status` | `notes` |
|---|---|---|
| E1, E2 from a rule, no flag | `rule_candidate` | what the span is |
| INC, P (not bias) | `rule_candidate_not_bias` | why it is not bias (guideline D3, D4) |
| I1, I2 (coded wording) | `for_review` | lexicon word alone; context-dependent; may be neutral job language |
| a flag is present (role noun, honorific, pronoun context, span too long) | `for_review` | the flag's meaning |
| `needs_review`, `E4_or_E5` | `for_review` | gender word the rules could not settle |
| `none` (rule rejected) | `rejected_by_rule` | no gender word in the span |

Unit-level notes are added for marital-status terms (out of gender scope in the guideline) and for pregnancy mentions (guideline D5: Others, with a note). These are notes only; they do not create spans, so no label overlaps another.

In [2]:
FLAG_NOTE = {
    "P_candidate_role_noun": "word is also an ordinary role noun (e.g. analyst, support, leader); may be Polisemantic (P)",
    "honorific_candidate": "honorific for a contact person; guideline D4 treats it as Polisemantic (P)",
    "needs_human_context": "pronoun or gender word whose generic/specific use depends on context",
    "span_narrowing_review": "span longer than the gender expression; the gendered words should be narrowed",
    "reject_candidate_no_gender_word": "rule matched a span without a gender word",
}
CODE_NOTE = {
    "E1": "stated gender restriction or preference",
    "E2": "gender-marked job title or word",
    "INC": "both genders listed; not bias (guideline D3)",
    "P": "not bias (guideline D4)",
    "I1": "masculine-coded lexicon word; single word, context-dependent; may be neutral job language",
    "I2": "feminine-coded lexicon word; single word, context-dependent; may be neutral job language",
    "E4_or_E5": "generic he / generic she undecided",
    "needs_review": "gender word the rules could not classify",
    "none": "",
}


def status_notes(r):
    code, flag = r["v02_candidate"], ("" if pd.isna(r["review_flag"]) else r["review_flag"])
    notes = [CODE_NOTE.get(code, "")]
    if flag:
        notes.append(FLAG_NOTE.get(flag, flag))
    if code == "none":
        return "rejected_by_rule", "; ".join(n for n in notes if n)
    if code in ("I1", "I2", "needs_review", "E4_or_E5") or flag:
        return "for_review", "; ".join(n for n in notes if n)
    if code in ("INC", "P"):
        return "rule_candidate_not_bias", "; ".join(n for n in notes if n)
    return "rule_candidate", "; ".join(n for n in notes if n)


res = cw.apply(status_notes, axis=1, result_type="expand")
spans = cw.copy()
spans["review_status"], spans["notes"] = res[0], res[1]
spans = spans.merge(units[["unit_id", "employer_id"]], on="unit_id", how="left")
display(pd.crosstab(spans["v02_candidate"], spans["review_status"], margins=True))

review_status,for_review,rejected_by_rule,rule_candidate,rule_candidate_not_bias,All
v02_candidate,,,,,
E1,1,0,316,0,317
E2,0,0,40,0,40
E4_or_E5,5,0,0,0,5
I1,350,0,0,0,350
I2,379,0,0,0,379
INC,0,0,0,99,99
P,11,0,0,0,11
needs_review,16,0,0,0,16
none,0,1,0,0,1


In [3]:
MARITAL = re.compile(r"\b(?:single|married|marital|civil status|widow\w*|separated)\b", re.I)
PREGNANT = re.compile(r"\bpregnan\w*\b", re.I)

unit_notes = {}
for r in units.itertuples():
    n = []
    if MARITAL.search(str(r.text_masked)):
        n.append("marital-status term: outside gender scope (guideline); recorded in notes only")
    if PREGNANT.search(str(r.text_masked)):
        n.append("pregnancy mention: guideline D5 = Others (O) with a note")
    if n:
        unit_notes[r.unit_id] = "; ".join(n)

u = units.copy()
span_by_unit = spans.groupby("unit_id")
u["n_candidate_spans"] = u["unit_id"].map(span_by_unit.size()).fillna(0).astype(int)
u["candidate_codes"] = u["unit_id"].map(span_by_unit["v02_candidate"].apply(lambda s: "|".join(sorted(set(s))))).fillna("")
u["unit_status"] = u["candidate_codes"].map(lambda c: "no_gender_expression_found" if c == "" else "has_candidate")
u["unit_notes"] = u["unit_id"].map(unit_notes).fillna("")
print(f"Units with unit-level notes: {(u['unit_notes'] != '').sum()} "
      f"(marital: {u['unit_notes'].str.contains('marital').sum()}, pregnancy: {u['unit_notes'].str.contains('pregnancy').sum()})")
display(u["unit_status"].value_counts().rename_axis("unit status").to_frame("units"))

Units with unit-level notes: 48 (marital: 47, pregnancy: 1)


,units
unit status,
no_gender_expression_found,6897
has_candidate,1074


## 2. Full-advertisement context

For each advertisement, the masked units are joined in order with their component tag, followed by the candidate spans. This is the context a later reviewer needs; the guideline path is stored in every row so the reference travels with the data.

In [4]:
GUIDE = "annotation-guidelines-gender-v0.2.md"
rows = []
spans_by_ad = spans.groupby("record_id")
for rid, g in u.sort_values(["record_id", "unit_index"]).groupby("record_id"):
    text = "\n".join(f"[{r.component}] {r.text_masked}" for r in g.itertuples())
    sp = spans_by_ad.get_group(rid) if rid in spans_by_ad.groups else spans.iloc[0:0]
    rows.append({
        "record_id": rid,
        "n_units": len(g),
        "ad_text_masked": text,
        "n_candidate_spans": len(sp),
        "candidate_codes": "|".join(sorted(set(sp["v02_candidate"]))),
        "n_for_review": int((sp["review_status"] == "for_review").sum()),
        "has_explicit_rule_candidate": bool(((sp["v02_family"] == "explicit") & (sp["review_status"] == "rule_candidate")).any()),
        "unit_notes": " | ".join(f"{x.unit_id}: {x.unit_notes}" for x in g.itertuples() if x.unit_notes),
        "guideline": GUIDE,
    })
ad_ctx = pd.DataFrame(rows)
print(f"{len(ad_ctx):,} ads; {(ad_ctx['n_candidate_spans'] > 0).sum():,} with at least one candidate span.")
display(ad_ctx[["record_id", "n_units", "n_candidate_spans", "candidate_codes", "n_for_review"]].head(8))

1,500 ads; 765 with at least one candidate span.


,record_id,n_units,n_candidate_spans,candidate_codes,n_for_review
0,row-0002,4,0,,0
1,row-0003,5,1,INC,0
2,row-0004,1,0,,0
3,row-0005,1,0,,0
4,row-0006,1,0,,0
5,row-0007,1,0,,0
6,row-0008,1,0,,0
7,row-0009,1,0,,0


### Examples shown with full context

Four ads, each chosen as the first in the dataset (by id) that has the stated feature: an explicit rule candidate, an INC listing, a gendered title, and implicit-only candidates.

In [5]:
def first_ad(mask):
    return ad_ctx[mask].sort_values("record_id").iloc[0]["record_id"]

picks = {
    "explicit (E1) rule candidate": first_ad(ad_ctx["candidate_codes"].str.contains("E1")),
    "INC listing": first_ad(ad_ctx["candidate_codes"].str.contains("INC")),
    "gendered title (E2)": first_ad(ad_ctx["candidate_codes"].str.contains("E2")),
    "implicit-only candidates": first_ad(ad_ctx["candidate_codes"].str.fullmatch(r"I[12](\|I[12])?")),
}
for label, rid in picks.items():
    print("=" * 100)
    print(label, "->", rid)
    g = u[u["record_id"] == rid].sort_values("unit_index")
    for r in g.itertuples():
        mark = ""
        if r.unit_id in span_by_unit.groups:
            mark = "   <- " + "; ".join(f"{x.v02_candidate} '{x.span_text}' [{x.review_status}]" for x in span_by_unit.get_group(r.unit_id).itertuples())
        print(f"[{r.component}] {r.text_masked}{mark}")

explicit (E1) rule candidate -> row-0022
[job_title] Visual Artist
[qualification] Male   <- E1 'Male' [rule_candidate]
[qualification] Graduate of any four (4) year Computer or Drafting related courses
[qualification] With skills in designing
INC listing -> row-0003
[job_title] Human Resource Assistant
[qualification] Female/Male   <- INC 'Female/Male' [rule_candidate_not_bias]
[qualification] Graduate of BS Psychology, BSBA Human Resource Management, BS ~Behavioral Science or other related courses
[qualification] With at least 1 year experience in human resources functions
[qualification] With knowledge in implementing all human resources faucet
gendered title (E2) -> row-0083
[job_title] Watchman   <- E2 'Watchman' [rule_candidate]
[qualification] Elementary School Graduate
implicit-only candidates -> row-0033
[job_title] Direct Sales Agent
[qualification] Willing to learn and work on field
[qualification] Hardworking, loyal, and trustworthy   <- I2 'loyal' [for_review]; I2 'trustwo

## 3. Summary of what the rules flag

Counts describe rule output. The same wording recurs across ads, so distinct texts are shown next to span rows.

In [6]:
status_summary = (
    spans.groupby(["v02_candidate", "review_status"])
    .agg(span_rows=("unit_id", "size"), units=("unit_id", "nunique"), ads=("record_id", "nunique"), distinct_span_texts=("span_text", lambda s: s.str.lower().nunique()))
    .reset_index()
)
display(status_summary)

top = (
    spans.assign(span_lc=spans["span_text"].str.lower())
    .groupby(["v02_candidate", "span_lc"]).size().rename("span_rows").reset_index()
    .sort_values(["v02_candidate", "span_rows"], ascending=[True, False])
    .groupby("v02_candidate").head(8)
)
display(top)

,v02_candidate,review_status,span_rows,units,ads,distinct_span_texts
0,E1,for_review,1,1,1,1
1,E1,rule_candidate,316,311,310,10
2,E2,rule_candidate,40,40,32,6
3,E4_or_E5,for_review,5,3,2,3
4,I1,for_review,350,324,240,41
5,I2,for_review,379,324,275,40
6,INC,rule_candidate_not_bias,99,99,98,5
7,P,for_review,11,11,11,1
8,needs_review,for_review,16,16,16,3
9,none,rejected_by_rule,1,1,1,1


,v02_candidate,span_lc,span_rows
4,E1,male,192
0,E1,female,100
10,E1,preferably male,7
9,E1,preferably female,6
6,E1,male only,3
2,E1,for female,2
3,E1,for male,2
5,E1,male applicant,2
12,E2,foreman,22
15,E2,salesman,13


## 4. Validation

In [7]:
checks = []


def rec(name, ok, detail=""):
    checks.append({"check": name, "passed": bool(ok), "detail": detail})


contact = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+|https?://|www\.|(?:\+?\d[\d\s().-]{8,}\d)", re.I)
rec("all 1,500 ads present in the context file", ad_ctx["record_id"].nunique() == ads["record_id"].nunique() == 1500, f"{len(ad_ctx)} ads")
rec("every unit appears once with a status", u["unit_id"].is_unique and len(u) == len(units) and u["unit_status"].notna().all(), f"{len(u):,} units")
rec("every crosswalk span row is carried over", len(spans) == len(cw), f"{len(spans):,} spans")
rec("every span belongs to an existing unit", spans["unit_id"].isin(units["unit_id"]).all())
rec("every span has a review status and, unless rejected, a note", spans["review_status"].notna().all() and (spans.loc[spans["review_status"] != "rejected_by_rule", "notes"] != "").all())
rec("all I1/I2 spans are marked for_review", bool((spans.loc[spans["v02_candidate"].isin(["I1", "I2"]), "review_status"] == "for_review").all()))
rec("labels belong to the schema or a named review state", set(spans["v02_candidate"]) <= set(schema) | {"needs_review", "E4_or_E5", "none"}, str(sorted(set(spans["v02_candidate"]))))
rec("units in the context text equal units in the table", ad_ctx["n_units"].sum() == len(units))
bad_contact = ad_ctx["ad_text_masked"].map(lambda t: bool(contact.search(t))).sum()
rec("no email, URL or phone-like string in the context text", bad_contact == 0, f"{bad_contact} ads matched")
rec("guideline file exists next to the notebook", Path(GUIDE).exists())
validation = pd.DataFrame(checks)
display(validation)

,check,passed,detail
0,"all 1,500 ads present in the context file",True,1500 ads
1,every unit appears once with a status,True,"7,971 units"
2,every crosswalk span row is carried over,True,"1,218 spans"
3,every span belongs to an existing unit,True,
4,"every span has a review status and, unless rejected, a note",True,
5,all I1/I2 spans are marked for_review,True,
6,labels belong to the schema or a named review state,True,"['E1', 'E2', 'E4_or_E5', 'I1', 'I2', 'INC', 'P', 'needs_review', 'none']"
7,units in the context text equal units in the table,True,
8,"no email, URL or phone-like string in the context text",True,0 ads matched
9,guideline file exists next to the notebook,True,


## 5. Export

In [8]:
span_cols = ["unit_id", "record_id", "employer_id", "component", "text_masked", "span_text", "span_start", "span_end", "span_offset_valid",
             "raw_start", "raw_end", "v02_candidate", "v02_family", "review_status", "review_flag", "notes", "gender_expression_type", "rule_id"]
spans[span_cols].to_csv(OUT / "preannotation_spans.csv", index=False, encoding="utf-8-sig")
u[["unit_id", "record_id", "component", "unit_index", "text_masked", "unit_status", "candidate_codes", "n_candidate_spans", "unit_notes"]].to_csv(OUT / "preannotation_units.csv", index=False, encoding="utf-8-sig")
ad_ctx.to_csv(OUT / "ad_context.csv", index=False, encoding="utf-8-sig")
validation.to_csv(OUT / "validation_checks.csv", index=False, encoding="utf-8-sig")
status_summary.to_csv(OUT / "status_summary.csv", index=False, encoding="utf-8-sig")
assert validation["passed"].all(), validation[~validation["passed"]]
print("Saved:", *sorted(str(p) for p in OUT.iterdir()), sep="\n  ")

Saved:
  outputs\s5_preannotation\ad_context.csv
  outputs\s5_preannotation\preannotation_spans.csv
  outputs\s5_preannotation\preannotation_units.csv
  outputs\s5_preannotation\status_summary.csv
  outputs\s5_preannotation\validation_checks.csv
